<a href="https://colab.research.google.com/github/apeas1/PIDDeteccionVehiculos/blob/main/Entrega_Final/Entrenamiento_Vehiculos_YOLOv8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Entrenamiento de modelos

In [1]:
!pip install -qq ultralytics
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
!pip install -qq roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="oLgrOUnp0TT0CVeH8mYq")
project = rf.workspace("sardar-vallabhbhai-national-institute-if-technology").project("vehicle-detection-wagr7")
version = project.version(3)
dataset = version.download("yolov8-obb")


dataset = version.download("yolov8")
print("Dataset descargado en:", dataset.location)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 58.2 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to VEHICLE-DETECTION-3 in yolov8-obb:: 100%|██████████| 7420/7420 [00:04<00:00, 1758.82it/s]


Dataset descargado en: /content/VEHICLE-DETECTION-3


In [3]:
%%writefile dataset.yaml
path: VEHICLE-DETECTION-3
train: train/images
val: valid/images
test: test/images

names:
  0: 3W
  1: Bus
  2: Coche
  3: Camioneta
  4: Motocicleta
  5: Camión Abierto
  6: Camion


Writing dataset.yaml


Entrenamiento del modelo

In [ ]:
model = YOLO("yolov8n.pt")  # modelo base ligero
model.conf = 0.45  # Nuevo umbral global
model.iou = 0.35 # Umbral global NMS


model.train(
    data="dataset.yaml",    #El shuffle es automático
    epochs=20,
    imgsz=640,              #Normalizacion de píxeles(parte del preprocesado junto a imgsz) es automática
    batch=10,
    augment=True
)


Métricas del modelo final y evolución del modelo durante el entrenamiento

In [ ]:
metrics = model.val(data="dataset.yaml")
df = pd.read_csv("runs/detect/train/results.csv")

In [ ]:

print("\n===== MÉTRICAS PRINCIPALES =====")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision media:", metrics.box.mp)
print("Recall media:", metrics.box.mr)




print("\n===== EVOLUCIÓN DEL MODELO SOBRE EL CONJUNTO DE VALIDACIÓN =====")
plt.plot(df['epoch'], df['metrics/precision(B)'], label='Precision')
plt.plot(df['epoch'], df['metrics/recall(B)'], label='Recall')
plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50')
plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
plt.xlabel('Epoch')
plt.ylabel('Valor')
plt.legend()
plt.show()
